# Part III exercises — the forward problem

About 30 minutes. These are heavier than Parts I and II: keep `n = 6` (a $64\times64$
grid) unless a cell says otherwise, or you will spend the session watching a progress
bar.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import qtade_cfd as cfd
import qtade_quimb as qq
import qtade_tn as tn

plt.rcParams.update({"figure.figsize": (8, 3), "axes.grid": True, "grid.alpha": 0.3})

n = 6
N = 2 ** n
xx = np.linspace(0, 1, N, endpoint=False)
X, Y = np.meshgrid(xx, xx, indexing="ij")
psi = 0.05 * np.sin(2 * np.pi * X) * np.sin(2 * np.pi * Y)


def initial_velocity():
    return (qq.from_grid(np.gradient(psi, xx, axis=1), eps=1e-9),
            qq.from_grid(-np.gradient(psi, xx, axis=0), eps=1e-9))

---
## Exercise 1 — what the nonlinearity actually costs

**Point:** the advection term is the only place where ranks *multiply* rather than add,
and it is worth knowing by how much.

In `qtade_cfd.Flow.step`, the advection term is `u * (Dx u) + v * (Dy u)`. Instrument it:
for one timestep, record the bond dimension

* of `u`,
* of `Dx u` after rounding,
* of the Hadamard product `u * (Dx u)` **before** rounding,
* and after rounding.

Predict the third number before you measure it.

In [ ]:
flow = cfd.Flow(n, nu=2e-3, dt=0.4 / N, chi_max=32, cutoff=1e-7)
u, v = initial_velocity()

# TODO: measure the four bond dimensions described above.
# tn.mpo_apply, tn.tt_hadamard and tn.tt_round are the pieces you need.

Then: the product $u\,\partial_x u$ can also be written $\tfrac12 \partial_x(u^2)$.
Both are one Hadamard and one MPO application. Is one of them cheaper in rank? Measure
rather than argue. (This matters: it is the difference between the advective and
conservative forms, and in a truncated setting they are *not* equivalent.)

---
## Exercise 2 — how much rank does the constraint need?

**Point:** truncation is optimal in the Frobenius norm, and the Frobenius norm knows
nothing about $\nabla\cdot\mathbf{v} = 0$.

Run 20 timesteps at `chi_max` in $\{8, 16, 32, 48\}$ and plot
$\|\nabla\cdot\mathbf{v}\|$ against step for each. Find the smallest `chi_max` that
keeps the divergence below $10^{-1}$ for the whole run, and say what that costs in
memory relative to the dense field.

In [ ]:
# TODO

Follow-up worth five minutes of discussion: could you *project back* onto the
divergence-free manifold after each rounding, instead of tightening `chi_max`? What
would that cost, and would the result still be a tensor train of bounded rank?

---
## Exercise 3 — put an obstacle in the flow

**Point:** geometry enters as one extra Hadamard product per step, and nothing else.

Build the smoothed mask of a disc from Part II, and apply it to both velocity
components at the end of every timestep. Then:

1. check that the velocity really is (nearly) zero inside the disc;
2. measure how much the mask raises the bond dimension of the flow;
3. vary $\alpha$ and see how sharp an obstacle you can afford.

In [ ]:
# Sign convention matters here and is easy to get backwards. The level set q must be
# NEGATIVE inside the solid, so that q + |q| vanishes there and the mask is exactly zero
# in the body. Flip the sign and you will faithfully simulate a flow that exists only
# inside the cylinder.
q = ((X - 0.35) ** 2 + (Y - 0.5) ** 2) - 0.15 ** 2      # < 0 inside the disc
inside = q < 0


def smoothed_mask(alpha):
    return qq.from_grid(1 - np.exp(-alpha * (q + np.abs(q))), eps=1e-6)


# TODO: run the flow with `u = round(hadamard(mask, u))` after each step.

---
## Exercise 4 — the spectrum is the easy diagnostic

**Point:** a compressed field can look perfectly healthy in the energy spectrum while
its higher-order statistics have been destroyed. If you only ever plot $E(k)$, you will
not notice.

Take the synthetic multiscale field below. Compress it to bond dimension
$\chi \in \{4, 8, 16, 32, 64\}$ and for each compare, against the original:

* the relative $L_2$ error,
* the energy spectrum $E(k)$,
* the **flatness** of one-cell velocity increments,
  $F = \langle \delta u^4\rangle / \langle \delta u^2\rangle^2$
  (3 for a Gaussian field; larger means intermittent).

In [ ]:
m = 8
Nm = 2 ** m
rng = np.random.default_rng(3)
kf = np.fft.fftfreq(Nm) * Nm
KX, KY = np.meshgrid(kf, kf, indexing="ij")
K = np.hypot(KX, KY)
K[0, 0] = 1.0
amp = K ** (-5 / 6)
amp[K > Nm / 3] = 0.0
ph = rng.standard_normal((Nm, Nm)) + 1j * rng.standard_normal((Nm, Nm))
base = np.real(np.fft.ifft2(amp * ph))
mult = np.ones((Nm, Nm))
for lvl in range(1, 7):                       # a multiplicative cascade adds intermittency
    s = 2 ** lvl
    mult *= np.kron(rng.lognormal(0, 0.35, (s, s)), np.ones((Nm // s, Nm // s)))
field = base * mult
field /= field.std()


def flatness(f, lag=1):
    d = (np.roll(f, -lag, axis=0) - f).ravel()
    return (d ** 4).mean() / (d ** 2).mean() ** 2


def spectrum(f):
    F = np.abs(np.fft.fft2(f)) ** 2
    kb = np.round(K).astype(int)
    return np.bincount(kb.ravel(), F.ravel())[: Nm // 2]


# TODO

---
## Solutions

In [ ]:
# --- Exercise 1 ---
flow = cfd.Flow(n, nu=2e-3, dt=0.4 / N, chi_max=32, cutoff=1e-7)
u, v = initial_velocity()
du = tn.tt_round(tn.mpo_apply(flow.Dx, u), eps=1e-7)
raw = tn.tt_hadamard(u, du)
rounded = tn.tt_round(raw, eps=1e-7)
print(f"chi(u)              = {max(tn.tt_ranks(u))}")
print(f"chi(Dx u)           = {max(tn.tt_ranks(du))}")
print(f"chi(u * Dx u) raw   = {max(tn.tt_ranks(raw))}   "
      f"(= {max(tn.tt_ranks(u))} x {max(tn.tt_ranks(du))}, ranks multiply)")
print(f"chi(u * Dx u) round = {max(tn.tt_ranks(rounded))}")

adv = tn.tt_round(tn.mpo_apply(flow.Dx, tn.tt_round(tn.tt_hadamard(u, u), eps=1e-7)),
                  eps=1e-7)
print(f"\nconservative form, chi(d/dx (u^2)/2) = {max(tn.tt_ranks(adv))}")
print("Both forms cost one Hadamard and one MPO application, and they agree in exact")
print("arithmetic. Under truncation they do not: the conservative form rounds the square")
print("before differentiating, the advective form rounds the product after. Which is")
print("better depends on the field, and you should measure rather than assume.")

In [ ]:
# --- Exercise 2 ---
plt.figure(figsize=(8, 3.5))
for chi in (8, 16, 32, 48):
    f = cfd.Flow(n, nu=2e-3, dt=0.4 / N, chi_max=chi, cutoff=1e-9, poisson_sweeps=1)
    uu, vv = initial_velocity()
    hist = []
    for _ in range(20):
        uu, vv, _ = f.step(uu, vv)
        hist.append(tn.tt_norm(cfd.divergence(f, uu, vv)))
    plt.semilogy(hist, label=f"$\\chi_{{max}}={chi}$")
    print(f"chi_max={chi:>3}: final |div| = {hist[-1]:.2e}, "
          f"params = {tn.tt_size(uu):>7,d} vs dense {N**2:,d} "
          f"({N**2 / tn.tt_size(uu):>5.1f}x)")
plt.xlabel("step"), plt.ylabel(r"$\|\nabla\cdot\mathbf{v}\|$"), plt.legend(fontsize=8)
plt.tight_layout()
print("""
Note the last column, and do not skip past it. On a 64x64 grid a train with chi = 48
stores MORE numbers than the dense field. Compression is an asymptotic statement:
n*chi^2 beats 4^n only once n is large enough, and at this toy resolution it is not.
Everything else in this notebook is still true and still worth measuring -- but if
someone shows you a compression factor, ask what grid it was measured on.""")

In [ ]:
# --- Exercise 3 ---
for alpha in (200, 2000):
    mask = smoothed_mask(alpha)
    f = cfd.Flow(n, nu=2e-3, dt=0.4 / N, chi_max=40, cutoff=1e-8, poisson_sweeps=1)
    uu, vv = initial_velocity()
    for _ in range(15):
        uu, vv, _ = f.step(uu, vv)
        uu = tn.tt_round(tn.tt_hadamard(mask, uu), eps=1e-8, chi_max=40)
        vv = tn.tt_round(tn.tt_hadamard(mask, vv), eps=1e-8, chi_max=40)
    grid = qq.to_grid(uu)
    print(f"alpha={alpha:>5}: chi(mask)={max(tn.tt_ranks(mask)):>3}  "
          f"chi(u)={max(tn.tt_ranks(uu)):>3}  "
          f"max|u| inside the disc = {np.abs(grid[inside]).max():.2e}  "
          f"(typical |u| outside: {np.abs(grid[~inside]).mean():.2e})")

plt.figure(figsize=(4, 3.5))
plt.imshow(qq.to_grid(uu).T, origin="lower", cmap="RdBu_r")
plt.contour(np.asarray(inside, float).T, levels=[0.5], colors="k", linewidths=1)
plt.xticks([]), plt.yticks([]), plt.title("$u$ with a masked obstacle")
plt.tight_layout()

In [ ]:
# --- Exercise 4 ---
ref_spec = spectrum(field)
print(f"reference flatness = {flatness(field):.2f}   (3.0 would be Gaussian)\n")
plt.figure(figsize=(8, 3.5))
kk = np.arange(1, Nm // 2)
plt.loglog(kk, ref_spec[1:], "k", lw=2, label="original")
for chi in (4, 8, 16, 32, 64):
    c = qq.from_grid(field, eps=1e-14, chi_max=chi)
    g = qq.to_grid(c)
    print(f"chi={chi:>3}: L2 err = {np.linalg.norm(g - field) / np.linalg.norm(field):.3f}"
          f"   flatness = {flatness(g):>6.2f}")
    plt.loglog(kk, spectrum(g)[1:], lw=1, label=f"$\\chi={chi}$")
plt.xlabel("k"), plt.ylabel("E(k)"), plt.legend(fontsize=8)
plt.tight_layout()
print("""
Compare the three columns as chi rises. At chi = 32 the spectrum is already within
about 13% across the inertial range while the flatness is still off by 70% and the
pointwise L2 error is 42%. At chi = 64 the spectrum is accurate to 1.4% and the
flatness is still 16% high.

That ordering is the warning Pisoni et al. (2026) give in 3D. E(k) is the diagnostic
that survives compression longest, which makes it the one least able to tell you
whether the compression was safe. Judge a compressed turbulent field on its increment
statistics, not on its spectrum.""")